<a href="https://colab.research.google.com/github/team0243/Project_ML/blob/main/RCC_UCUT_RFE_TUNEPARAMETER_TRAIN_EVALUATE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing the Python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')
from sklearn.tree import plot_tree

In [ ]:
df = pd.read_excel('Dataset_RCC_UTUC_ML2.xlsx') # replace .xlsx file

In [ ]:
# Prints information about a DataFrame
df.info()

In [ ]:
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

In [ ]:
X.head(5)

In [ ]:
X.shape, y.shape

In [ ]:
# To solve the imbalance problem between categories 0 and 1.
# Apply SMOTE (Synthetic Minority Oversampling Technique) – Oversampling

sm = SMOTE(sampling_strategy = 0.90 ,random_state = 25)
X_resampled, y_resampled = sm.fit_resample(X,y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size = 0.3, random_state = 25)

In [ ]:
y_train.value_counts(), y_test.value_counts()

In [ ]:
n_features = [8,7,6,5,4,2]

model = LogisticRegression()
#model = MLPClassifier()
column_names = X.columns.tolist()

In [ ]:
# Iterate over the n_features_to_select values
for n in n_features:
    rfe = RFE(estimator = model, n_features_to_select = n)

    rfe.fit(X_train, y_train)

    selected_feature_indices = [i for i, support in enumerate(rfe.support_) if support]

    X_train_selected = X_train.iloc[:, selected_feature_indices]
    X_test_selected = X_test.iloc[:, selected_feature_indices]

    model.fit(X_train_selected, y_train)

    y_pred = model.predict(X_test_selected)

    # Calculate the accuracy score
    accuracy = accuracy_score(y_test, y_pred)

    # Print the selected features and accuracy
    print(f"Number of Selected Features = {n}")
    print("Selected Features")
    for col in X_train_selected.columns:
        print(col)
    print("-------------------------------------------")
    print("Accuracy:", accuracy)
    print("-------------------------------------------")
    print("Classification report")
    print()
    print(classification_report(y_test, y_pred))
    print("-------------------------------------------")

# **KNeighborsClassifier**

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2

n_features = [8,7,6,5,4,2]

# Use SelectKBest with chi2 scoring function
for n in n_features:
    selector = SelectKBest(chi2, k=n)
    X_train_selected_knn = selector.fit_transform(X_train, y_train)
    X_test_selected_knn = selector.transform(X_test)


    modelknn = KNeighborsClassifier()
    modelknn.fit(X_train_selected_knn, y_train)
    y_pred_knn = modelknn.predict(X_test_selected_knn)


    accuracy = accuracy_score(y_test, y_pred_knn)

    # Print the selected features and accuracy
    print(f"Number of Selected Features = {n}")
    print("Selected Features")
    # Get the indices of the selected features
    selected_feature_indices = selector.get_support(indices=True)
    # Get the names of the selected features from the original DataFrame
    selected_feature_names = X.columns[selected_feature_indices]
    # Print the selected feature names
    for col in selected_feature_names:
        print(col)
    print("-------------------------------------------")
    print("Accuracy:", accuracy)
    print("-------------------------------------------")
    print("Classification report")
    print()
    print(classification_report(y_test, y_pred_knn))
    print("-------------------------------------------")

# SVM classifier RFE

In [ ]:
# prompt: create model SVM classifier

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')
# Load the dataset
df = pd.read_excel('Dataset_RCC_UTUC_ML2.xlsx')  # replace .xlsx file

# Separate features (X) and target (y)
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

# Apply SMOTE for oversampling
sm = SMOTE(sampling_strategy=0.90, random_state=25)
X_resampled, y_resampled = sm.fit_resample(X, y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=25)

# Feature selection with SelectKBest
n_features = [8, 7, 6, 5, 4, 2]

# SVM model
for n in n_features:
  selector = SelectKBest(chi2, k=n)
  X_train_selected_svm = selector.fit_transform(X_train, y_train)
  X_test_selected_svm = selector.transform(X_test)

  #model_svm = SVC()
  # Hyperparameter tuning with GridSearchCV
  param_grid = {'C': [0.1, 1, 10, 100], 'kernel': ['linear', 'rbf', 'poly'], 'gamma': ['scale', 'auto']}
  grid_svm = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3)
  grid_svm.fit(X_train_selected_svm,y_train)
  print("Best Parameters:", grid_svm.best_params_)
  model_svm = grid_svm.best_estimator_

  # Train the SVM model
  model_svm.fit(X_train_selected_svm, y_train)
  y_pred_svm = model_svm.predict(X_test_selected_svm)
  accuracy = accuracy_score(y_test, y_pred_svm)

  # Print the selected features and accuracy
  print(f"Number of Selected Features = {n}")
  print("Selected Features")
  selected_feature_indices = selector.get_support(indices=True)
  selected_feature_names = X.columns[selected_feature_indices]
  for col in selected_feature_names:
      print(col)
  print("-------------------------------------------")
  print("Accuracy:", accuracy)
  print("-------------------------------------------")
  print("Classification report")
  print()
  print(classification_report(y_test, y_pred_svm))
  print("-------------------------------------------")

# Logistic Regression

In [ ]:
# prompt: create model logistic regression classifier

import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')

# Load the dataset
try:
    df = pd.read_excel('Dataset_RCC_UTUC_ML2.xlsx')  # replace .xlsx file
except FileNotFoundError:
  print('Dataset_RCC_UTUC_ML2.xlsx file not found. Upload it and retry.')
  exit()
# Separate features (X) and target (y)
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

# Apply SMOTE for oversampling
sm = SMOTE(sampling_strategy=0.95, random_state=25)
X_resampled, y_resampled = sm.fit_resample(X, y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=25)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model training with all features
model = LogisticRegression(random_state=25)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("-------------------------------------------")
print("Logistic Regression Model with all features")
print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("-------------------------------------------")



In [ ]:
#RFE feature selection
n_features_range = [2, 4, 6, 7, 8]
for n_features_to_select in n_features_range:
  rfe = RFE(estimator=LogisticRegression(random_state=25), n_features_to_select=n_features_to_select)
  X_train_rfe = rfe.fit_transform(X_train, y_train)
  X_test_rfe = rfe.transform(X_test)

  # Retrain the model with selected features.
  model = LogisticRegression(random_state=25)
  model.fit(X_train_rfe, y_train)
  y_pred_rfe = model.predict(X_test_rfe)
  accuracy_rfe = accuracy_score(y_test, y_pred_rfe)

  print(f"Logistic Regression Model with {n_features_to_select} Features (RFE)")
  print("-------------------------------------------")
  print(f"Accuracy: {accuracy_rfe}")
  print("Classification Report:")
  print(classification_report(y_test, y_pred_rfe))
  print("-------------------------------------------")


# create model Decision Tree classifier

In [ ]:
# prompt: create model Decision Tree classifier

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings

warnings.filterwarnings('ignore')

try:
  df = pd.read_excel('Dataset_RCC_UTUC_ML2.xlsx')  # replace .xlsx file
except FileNotFoundError:
  print('Dataset_RCC_UTUC_ML2.xlsx file not found. Upload it and retry.')
  exit()
# Separate features (X) and target (y)
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

# Apply SMOTE for oversampling
sm = SMOTE(sampling_strategy=0.85, random_state=25)
X_resampled, y_resampled = sm.fit_resample(X, y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=25)
# Create a Decision Tree classifier
model_dt = DecisionTreeClassifier(random_state=25)

# Define a grid of hyperparameters to tune
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Use GridSearchCV to find the best hyperparameters
grid_dt = GridSearchCV(model_dt, param_grid, cv=5, scoring='accuracy')
grid_dt.fit(X_train, y_train)

# Print the best hyperparameters found
print("Best Hyperparameters:", grid_dt.best_params_)

# Use the best model found by GridSearchCV
best_model_dt = grid_dt.best_estimator_

# Train the model on the full training set
best_model_dt.fit(X_train, y_train)

# Make predictions on the test set
y_pred_dt = best_model_dt.predict(X_test)

# Evaluate the model
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("-------------------------------------------")
print("Decision Tree Classifier with best hyperparameters")
print(f"Accuracy: {accuracy_dt}")
print("Classification Report:")
print(classification_report(y_test, y_pred_dt))
print("-------------------------------------------")


In [ ]:
#Feature selection with RFE
n_features_range = [2, 4, 6, 7, 8]

for n_features_to_select in n_features_range:
  rfe = RFE(estimator=DecisionTreeClassifier(random_state=25), n_features_to_select=n_features_to_select)
  X_train_rfe = rfe.fit_transform(X_train, y_train)
  X_test_rfe = rfe.transform(X_test)

  # Retrain the model with selected features.
  model = DecisionTreeClassifier(random_state=25)
  model.fit(X_train_rfe, y_train)
  y_pred_rfe = model.predict(X_test_rfe)

  accuracy_rfe = accuracy_score(y_test, y_pred_rfe)

  print(f"Decision Tree Model with {n_features_to_select} Features (RFE)")
  print("-------------------------------------------")
  print(f"Accuracy: {accuracy_rfe}")
  print("Classification Report:")
  print(classification_report(y_test, y_pred_rfe))
  print("-------------------------------------------")


**KNeighborsClassifier**

In [ ]:
# prompt: create model KNeighbors Classifier

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, chi2
import warnings
warnings.filterwarnings('ignore')

try:
    df = pd.read_excel('Dataset_RCC_UTUC_ML2.xlsx')  # replace .xlsx file
except FileNotFoundError:
    print('Dataset_RCC_UTUC_ML2.xlsx file not found. Upload it and retry.')
    exit()

X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

sm = SMOTE(sampling_strategy=0.90, random_state=25)
X_resampled, y_resampled = sm.fit_resample(X, y)
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=25)

n_features = [8, 7, 6, 5, 4, 2]

for n in n_features:
  selector = SelectKBest(chi2, k=n)
  X_train_selected_knn = selector.fit_transform(X_train, y_train)
  X_test_selected_knn = selector.transform(X_test)

  modelknn = KNeighborsClassifier()
  modelknn.fit(X_train_selected_knn, y_train)
  y_pred_knn = modelknn.predict(X_test_selected_knn)

  accuracy = accuracy_score(y_test, y_pred_knn)

  print(f"Number of Selected Features = {n}")
  print("Selected Features")
  selected_feature_indices = selector.get_support(indices=True)
  selected_feature_names = X.columns[selected_feature_indices]
  for col in selected_feature_names:
    print(col)
  print("-------------------------------------------")
  print("Accuracy:", accuracy)
  print("-------------------------------------------")
  print("Classification report")
  print()
  print(classification_report(y_test, y_pred_knn))
  print("-------------------------------------------")


# Random forest cassifier

In [ ]:
# prompt: create model Random forest cassifier  and GridSearchCV and RFE

import pandas as pd
import numpy as np
import warnings
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')

try:
  df = pd.read_excel('Dataset_RCC_UTUC_ML2.xlsx')  # replace .xlsx file
except FileNotFoundError:
  print('Dataset_RCC_UTUC_ML2.xlsx file not found. Upload it and retry.')
  exit()
# Separate features (X) and target (y)
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

# Apply SMOTE for oversampling
sm = SMOTE(sampling_strategy=0.90, random_state=25)
X_resampled, y_resampled = sm.fit_resample(X, y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=25)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create a Random Forest classifier
model_rf = RandomForestClassifier(random_state=25)

# Define a grid of hyperparameters to tune
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Use GridSearchCV to find the best hyperparameters
grid_rf = GridSearchCV(model_rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_rf.fit(X_train, y_train)

# Print the best hyperparameters found
print("Best Hyperparameters:", grid_rf.best_params_)

# Use the best model found by GridSearchCV
best_model_rf = grid_rf.best_estimator_

# Train the model on the full training set
best_model_rf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_rf = best_model_rf.predict(X_test)

# Evaluate the model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("-------------------------------------------")
print("Random Forest Classifier with best hyperparameters")
print(f"Accuracy: {accuracy_rf}")
print("Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("-------------------------------------------")


In [ ]:
# Feature selection with RFE
n_features_range = [2, 4, 6, 7, 8]

for n_features_to_select in n_features_range:
  rfe = RFE(estimator=RandomForestClassifier(random_state=25, n_estimators=50), n_features_to_select=n_features_to_select)
  X_train_rfe = rfe.fit_transform(X_train, y_train)
  X_test_rfe = rfe.transform(X_test)

  # Retrain the model with selected features.
  model = RandomForestClassifier(random_state=25, n_estimators=50)
  model.fit(X_train_rfe, y_train)
  y_pred_rfe = model.predict(X_test_rfe)

  accuracy_rfe = accuracy_score(y_test, y_pred_rfe)

  print(f"Random Forest Model with {n_features_to_select} Features (RFE)")
  print("-------------------------------------------")
  print(f"Accuracy: {accuracy_rfe}")
  print("Classification Report:")
  print(classification_report(y_test, y_pred_rfe))
  print("-------------------------------------------")


# Gradian Booting Classifier

In [ ]:
# prompt:  Create model Gradian Booting Classifier  and RFE

import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')

try:
    df = pd.read_excel('Dataset_RCC_UTUC_ML2.xlsx')  # replace .xlsx file
except FileNotFoundError:
    print('Dataset_RCC_UTUC_ML2.xlsx file not found. Upload it and retry.')
    exit()

X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

sm = SMOTE(sampling_strategy=0.85, random_state=25)
X_resampled, y_resampled = sm.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=25)

model_gb = GradientBoostingClassifier(random_state=25)

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 3]
}

grid_gb = GridSearchCV(model_gb, param_grid, cv=5, scoring='accuracy', verbose=2)
grid_gb.fit(X_train, y_train)

print("Best Hyperparameters:", grid_gb.best_params_)

best_model_gb = grid_gb.best_estimator_

best_model_gb.fit(X_train, y_train)

y_pred_gb = best_model_gb.predict(X_test)

accuracy_gb = accuracy_score(y_test, y_pred_gb)
print("-------------------------------------------")
print("Gradient Boosting Classifier with best hyperparameters")
print(f"Accuracy: {accuracy_gb}")
print("Classification Report:")
print(classification_report(y_test, y_pred_gb))
print("-------------------------------------------")

In [ ]:
n_features_range = [2, 4, 6, 7, 8]

for n_features_to_select in n_features_range:
  rfe = RFE(estimator=GradientBoostingClassifier(random_state=25), n_features_to_select=n_features_to_select)
  X_train_rfe = rfe.fit_transform(X_train, y_train)
  X_test_rfe = rfe.transform(X_test)

  model = GradientBoostingClassifier(random_state=25)
  model.fit(X_train_rfe, y_train)
  y_pred_rfe = model.predict(X_test_rfe)

  accuracy_rfe = accuracy_score(y_test, y_pred_rfe)

  print(f"Gradient Boosting Model with {n_features_to_select} Features (RFE)")
  print("-------------------------------------------")
  print(f"Accuracy: {accuracy_rfe}")
  print("Classification Report:")
  print(classification_report(y_test, y_pred_rfe))
  print("-------------------------------------------")

In [ ]:
# prompt: create  ROC curves Gradient Boosting Model

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
import warnings

# Assuming 'UTUC' is the positive class
# Convert y_test to numerical values (0 and 1)
y_test_numeric = y_test.map({'RCC': 0, 'UTUC': 1})

# Calculate ROC Curve using the numerical y_test
y_prob_gb = best_model_gb.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test_numeric, y_prob_gb)
roc_auc = roc_auc_score(y_test_numeric, y_prob_gb)

# Plot ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) - Gradient Boosting')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# prompt: Create ROC curve with multi model on above except SVM model

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
import warnings

warnings.filterwarnings('ignore')

try:
  df = pd.read_excel('Dataset_RCC_UTUC_ML2.xlsx')  # replace .xlsx file
except FileNotFoundError:
  print('Dataset_RCC_UTUC_ML2.xlsx file not found. Upload it and retry.')
  exit()
# Separate features (X) and target (y)
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

# Apply SMOTE for oversampling
sm = SMOTE(sampling_strategy=0.90, random_state=25)
X_resampled, y_resampled = sm.fit_resample(X, y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=25)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Models and their names
models = {
    "Logistic Regression": LogisticRegression(random_state=25),
    "Decision Tree": DecisionTreeClassifier(random_state=25),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=25),
    "Gradient Boosting": GradientBoostingClassifier(random_state=25)
}

# Dictionary to store ROC data
roc_data = {}

# Iterate through models
for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Handle hyperparameter tuning for specific models
    if model_name == "Decision Tree":
        param_grid = {
            'criterion': ['gini', 'entropy'],
            'max_depth': [None, 5, 10, 15, 20],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
        grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
        grid_search.fit(X_train, y_train)
        model = grid_search.best_estimator_

    elif model_name == "Random Forest":
        param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2']
        }
        grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
        grid_search.fit(X_train, y_train)
        model = grid_search.best_estimator_

    elif model_name == "Gradient Boosting":
        param_grid = {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 4, 5],
            'min_samples_split': [2, 4, 6],
            'min_samples_leaf': [1, 2, 3]
        }
        grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', verbose=0)
        grid_search.fit(X_train, y_train)
        model = grid_search.best_estimator_

    # Train the model
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_test_numeric = y_test.map({'RCC': 0, 'UTUC': 1})
    fpr, tpr, thresholds = roc_curve(y_test_numeric, y_prob)
    roc_auc = roc_auc_score(y_test_numeric, y_prob)
    roc_data[model_name] = (fpr, tpr, roc_auc)

# Plotting the ROC curves
plt.figure(figsize=(10, 8))
for model_name, (fpr, tpr, roc_auc) in roc_data.items():
    plt.plot(fpr, tpr, lw=2, label=f'{model_name} (area = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.legend(loc="lower right")
plt.show()
